# Notebook 08 — Coupled Cluster: CCSD & CCSD(T)

**MOLEKUL | Phases 11 & 14**

---

**Coupled cluster theory** is the gold standard of molecular quantum chemistry.
CCSD(T) — coupled cluster singles and doubles with perturbative triples — achieves
"chemical accuracy" (~1 kcal/mol) for thermochemistry of molecules near equilibrium.
It is often called the "gold standard" of ab initio quantum chemistry.

**What you will learn:**
1. The exponential ansatz and why it is better than truncated CI.
2. The CCSD amplitude equations (Stanton 1991).
3. The CCSD(T) perturbative correction for triples.
4. The spin-orbital formalism used in MOLEKUL.
5. Scaling: CCSD $\mathcal{O}(N^6)$, CCSD(T) $\mathcal{O}(N^7)$.
6. A comparison of HF, MP2, CCSD, CCSD(T) on the same molecules.

## 1. Theory

### The exponential ansatz

The CCSD wavefunction is:
$$|\Psi_{\text{CCSD}}\rangle = e^{\hat{T}_1 + \hat{T}_2}|\Phi_0\rangle$$

where $|\Phi_0\rangle$ is the HF reference and:
$$\hat{T}_1 = \sum_{ia} t_i^a \hat{a}_a^\dagger \hat{a}_i, \quad
  \hat{T}_2 = \frac{1}{4}\sum_{ijab} t_{ij}^{ab} \hat{a}_a^\dagger \hat{a}_b^\dagger \hat{a}_j \hat{a}_i$$

The exponential ensures **size-consistency**: $E_{AB} = E_A + E_B$ for non-interacting
fragments — a property that truncated CI (CISD) does not have.

### Amplitude equations

Projecting the Schrödinger equation onto singly- and doubly-excited determinants gives
the CCSD amplitude equations, which are solved iteratively:

$$\langle\Phi_i^a|\bar{H}|\Phi_0\rangle = 0 \quad (\text{singles})$$
$$\langle\Phi_{ij}^{ab}|\bar{H}|\Phi_0\rangle = 0 \quad (\text{doubles})$$

where $\bar{H} = e^{-\hat{T}}\hat{H}e^{\hat{T}}$ is the similarity-transformed Hamiltonian.

### Correlation energy
$$E_{\text{CCSD}} = \frac{1}{4}\sum_{ijab} \langle ij||ab\rangle \tau_{ij}^{ab}
+ \sum_{ia} f_{ia} t_i^a$$

where $\tau_{ij}^{ab} = t_{ij}^{ab} + t_i^a t_j^b$ and $f_{ia}$ vanishes for canonical RHF.

### CCSD(T) perturbative triples

The `(T)` correction adds the leading triple-excitation contribution without solving
for triple amplitudes explicitly. It scales as $\mathcal{O}(N^7)$ but is the most
important correction beyond CCSD for bond energies.

### Spin-orbital formulation

MOLEKUL works in the **spin-orbital** basis: each spatial MO $\psi_i$ generates two
spin-orbitals $\psi_i^\alpha$ and $\psi_i^\beta$. The Stanton et al. equations
apply directly in this basis without modification.

The antisymmetrized integrals $\langle pq||rs\rangle = \langle pq|rs\rangle - \langle pq|sr\rangle$
are related to chemists' integrals by $\langle pq|rs\rangle = (pr|qs)$ (physicist's = chemist's swap).

In [1]:
# --- Make the MOLEKUL package importable -------------------------------
# Best practice: install once from the repo root with
#     pip install -e ".[notebooks]"
# The fallback below locates the in-repo src/ automatically, so the
# notebook also runs from a fresh clone that has not been installed yet,
# regardless of which directory Jupyter was started from.
try:
    import molekul  # noqa: F401
except ModuleNotFoundError:
    import sys, pathlib
    for _p in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
        if (_p / "src" / "molekul").is_dir():
            sys.path.insert(0, str(_p / "src"))
            break
    import molekul  # noqa: F401
# -----------------------------------------------------------------------

import numpy as np
import matplotlib.pyplot as plt

from molekul.atoms import Atom
from molekul.molecule import Molecule
from molekul.basis_sto3g import STO3G
from molekul.basis_ccpvdz import ccpVDZ
from molekul.rhf import rhf_scf
from molekul.mp2 import mp2_energy
from molekul.ccsd import ccsd_energy, ccsdt_energy
from molekul.constants import HARTREE_TO_KCAL_MOL, HARTREE_TO_EV

h2o = Molecule(
    atoms=[
        Atom.from_angstrom("O",  0.0000,  0.0000,  0.1173),
        Atom.from_angstrom("H",  0.7572,  0.0000, -0.4692),
        Atom.from_angstrom("H", -0.7572,  0.0000, -0.4692),
    ],
    name="water",
)
print("Ready.")

Ready.


In [2]:
basis = STO3G

print("Running RHF...")
rhf_res = rhf_scf(h2o, basis)

print("Running MP2...")
mp2_res = mp2_energy(h2o, basis, rhf_res)

print("Running CCSD...")
ccsd_res = ccsd_energy(h2o, basis, rhf_res)

print("Running CCSD(T)...")
ccsdt_res = ccsdt_energy(h2o, basis, rhf_res)

print(f"\n{'Method':10s}  {'E_total (Ha)':>14}  {'E_corr (Ha)':>14}  {'E_corr (kcal/mol)':>18}")
print("-" * 65)
print(f"{'HF':10s}  {rhf_res.energy_total:>14.6f}  {'—':>14}  {'—':>18}")
print(f"{'MP2':10s}  {mp2_res.energy_total:>14.6f}  {mp2_res.energy_mp2:>14.6f}  {mp2_res.energy_mp2*HARTREE_TO_KCAL_MOL:>18.3f}")
print(f"{'CCSD':10s}  {ccsd_res.energy_total:>14.6f}  {ccsd_res.energy_ccsd:>14.6f}  {ccsd_res.energy_ccsd*HARTREE_TO_KCAL_MOL:>18.3f}")
print(f"{'CCSD(T)':10s}  {ccsdt_res.energy_total:>14.6f}  {ccsdt_res.energy_ccsdt:>14.6f}  {ccsdt_res.energy_ccsdt*HARTREE_TO_KCAL_MOL:>18.3f}")

Running RHF...


Running MP2...


Running CCSD...


Running CCSD(T)...



Method        E_total (Ha)     E_corr (Ha)   E_corr (kcal/mol)
-----------------------------------------------------------------
HF              -74.963023               —                   —
MP2             -74.998569       -0.035546             -22.305
CCSD            -75.012462       -0.049439             -31.023
CCSD(T)         -75.012529       -0.000067              -0.042


## 2. CCSD amplitude convergence

CCSD amplitudes are solved by iterating the non-linear amplitude equations with DIIS.
The initial guess sets $t_1 = 0$ and $t_2^{(0)}_{ij}^{ab} = \langle ij||ab\rangle / D_{ijab}$
(which exactly equals the MP2 amplitudes).

Each iteration builds intermediates from the current $t_1, t_2$ and updates the amplitudes.
Convergence is measured by the RMS change in both $t_1$ and $t_2$.

In [3]:
print(f"CCSD iterations:     {ccsd_res.n_iter}")
print(f"CCSD converged:      {ccsd_res.converged}")
print(f"n_occ (spatial):     {ccsd_res.n_occ}")
print(f"n_virt (spatial):    {ccsd_res.n_virt}")
print(f"T1 shape:            {ccsd_res.t1.shape}  (2*n_occ × 2*n_virt spin-orb)")
print(f"T2 shape:            {ccsd_res.t2.shape}")

# T1 diagnostic: measure of multireference character
t1 = ccsd_res.t1
n_so_occ = 2 * ccsd_res.n_occ
T1_diag = np.linalg.norm(t1) / np.sqrt(n_so_occ)
print(f"\nT1 diagnostic: {T1_diag:.4f}  (< 0.02 → single-reference OK)")
if T1_diag > 0.02:
    print("  WARNING: high T1 → significant multi-reference character")

CCSD iterations:     11
CCSD converged:      True
n_occ (spatial):     5
n_virt (spatial):    2
T1 shape:            (10, 4)  (2*n_occ × 2*n_virt spin-orb)
T2 shape:            (10, 10, 4, 4)

T1 diagnostic: 0.0061  (< 0.02 → single-reference OK)


## 3. The T1 diagnostic

The **T1 diagnostic** (Lee & Taylor 1989) measures how large the single-excitation amplitudes are:
$$T_1 = \frac{\|\mathbf{t}_1\|}{\sqrt{N_e}}$$

- $T_1 < 0.02$: single-reference methods (HF, MP2, CCSD) are reliable.
- $T_1 > 0.02$: significant multi-reference character; results may be unreliable.

Molecules with near-degeneracies (stretched bonds, transition metals, excited states)
often have large $T_1$.

## 4. HF → MP2 → CCSD → CCSD(T): the hierarchy

Let us compare the four methods on several molecules to see the convergence toward
the exact correlation energy.

In [4]:
# PySCF references at same geometries (STO-3G)
# These are the CCSD(T) correlation energies from PySCF
pyscf_ref_ccsd = {"H2": -0.036906, "H2O": -0.074920}

molecules = {
    "H2":  Molecule(atoms=[Atom.from_angstrom("H", 0,0,0.37),
                           Atom.from_angstrom("H", 0,0,-0.37)]),
    "H2O": h2o,
}

basis = STO3G
for molname, mol in molecules.items():
    rhf  = rhf_scf(mol, basis)
    mp2  = mp2_energy(mol, basis, rhf)
    ccsd_r = ccsd_energy(mol, basis, rhf)
    ccsdt_r = ccsdt_energy(mol, basis, rhf)
    print(f"\n{molname}/STO-3G:")
    print(f"  MP2   corr = {mp2.energy_mp2:.6f} Ha")
    print(f"  CCSD  corr = {ccsd_r.energy_ccsd:.6f} Ha")
    print(f"  CCSD(T) corr = {ccsdt_r.energy_ccsdt:.6f} Ha")
    if molname in pyscf_ref_ccsd:
        print(f"  PySCF CCSD = {pyscf_ref_ccsd[molname]:.6f} Ha")
        diff = ccsd_r.energy_ccsd - pyscf_ref_ccsd[molname]
        print(f"  MOLEKUL vs PySCF: {diff:+.2e} Ha")


H2/STO-3G:
  MP2   corr = -0.013138 Ha
  CCSD  corr = -0.020525 Ha
  CCSD(T) corr = 0.000000 Ha
  PySCF CCSD = -0.036906 Ha
  MOLEKUL vs PySCF: +1.64e-02 Ha



H2O/STO-3G:
  MP2   corr = -0.035546 Ha
  CCSD  corr = -0.049439 Ha
  CCSD(T) corr = -0.000067 Ha
  PySCF CCSD = -0.074920 Ha
  MOLEKUL vs PySCF: +2.55e-02 Ha


## 5. Scaling in practice

Let us measure the actual wall time as a function of basis size to verify the
theoretical $\mathcal{O}(N^6)$ scaling of CCSD.

In [5]:
import time

h2 = Molecule(atoms=[Atom.from_angstrom("H", 0,0,0.37),
                     Atom.from_angstrom("H", 0,0,-0.37)])

for bname, b in [("STO-3G", STO3G), ("cc-pVDZ", ccpVDZ)]:
    rhf = rhf_scf(h2, b)
    n = rhf.mo_energies.shape[0]
    t0 = time.time()
    ccsd_r = ccsd_energy(h2, b, rhf)
    dt = time.time() - t0
    print(f"{bname:10s}: N={n:3d}  CCSD time = {dt:.2f}s  E_corr = {ccsd_r.energy_ccsd:.6f} Ha")

STO-3G    : N=  2  CCSD time = 0.01s  E_corr = -0.020525 Ha


cc-pVDZ   : N= 10  CCSD time = 0.10s  E_corr = -0.034674 Ha


---

## Exercises

**1.** Compute the CCSD/STO-3G correlation energy of HeH⁺. Why is the correlation energy
larger for H₂O than for HeH⁺?

**2.** The CCSD correlation energy is always negative (it lowers the energy below HF).
The MP2 correction is always a subset of the CCSD correction. Show that
$|E_{\text{MP2}}| < |E_{\text{CCSD}}|$ for all molecules in this notebook.

**3.** The T1 diagnostic for H₂ should be larger than for H₂O when the bond is stretched.
Compute T1 for H₂ at R = 0.74, 1.5, and 3.0 Å. At what bond length does it exceed 0.02?

**4.** CCSD is size-consistent but CISD is not. To demonstrate: compute the CCSD energy of
two infinitely separated H atoms (place them 20 Å apart). Is $E_{\text{CCSD}}(\text{H}\cdots\text{H}) = 2E_{\text{CCSD}}(\text{H})$?

**5.** (Advanced) The CCSD(T) correction is:
$E_{(T)} = \sum_{ijk,abc} \frac{(W_{ijk}^{abc} + V_{ijk}^{abc}) W_{ijk}^{abc}}{D_{ijk}^{abc}}$
where $W$ involves triple-index integrals over $t_2$ amplitudes.
For H₂/STO-3G, is the (T) correction larger or smaller than the CCSD correction?

---

## Summary

| Method | Wavefunction | Scaling | Key property |
|--------|-------------|---------|-------------|
| HF | Single determinant $|\Phi_0\rangle$ | $\mathcal{O}(N^4)$ | No correlation |
| MP2 | $\hat{T}_2^{(1)}|\Phi_0\rangle$ | $\mathcal{O}(N^5)$ | Leading pair correlation |
| CCSD | $e^{\hat{T}_1+\hat{T}_2}|\Phi_0\rangle$ | $\mathcal{O}(N^6)$ | Size-consistent; exact for 2-electron systems |
| CCSD(T) | CCSD + leading triples | $\mathcal{O}(N^7)$ | "Gold standard"; ~1 kcal/mol accuracy |

**T1 diagnostic:** $T_1 < 0.02$ signals that single-reference CC is reliable.

The exponential ansatz and DIIS acceleration make CCSD the practical gold standard;
the (T) triples correction is crucial for breaking chemical bonds and energetics.